# POWER_FAULT_ANALYZER — Experiment Runner
Este notebook ejecuta los módulos Python del paquete. La lógica de dominio permanece en `src/`.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "power_fault_analyzer").is_dir():
    raise RuntimeError("Open this notebook with the project root as the working directory.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from uuid import uuid4
from power_fault_analyzer.domain.aggregates.network import Network
from power_fault_analyzer.domain.entities.bus import Bus
from power_fault_analyzer.domain.entities.electrical_equipment import TwoTerminalEquipmentBase
from power_fault_analyzer.domain.entities.terminal import Terminal
from power_fault_analyzer.domain.enums.equipment_family import EquipmentFamily
from power_fault_analyzer.domain.services.network_validation_service import DefaultNetworkValidationService
from power_fault_analyzer.domain.value_objects.voltage import Voltage


In [2]:
class DemonstrationBranch(TwoTerminalEquipmentBase):
    def __init__(self, name: str, voltage_kv: float = 13.8) -> None:
        equipment_id = uuid4()
        voltage = Voltage.from_kv(voltage_kv)
        terminals = (
            Terminal(equipment_id, 'FROM', voltage),
            Terminal(equipment_id, 'TO', voltage),
        )
        super().__init__(
            id=equipment_id,
            name=name,
            terminals=terminals,
            family=EquipmentFamily.TRANSMISSION,
        )


In [3]:
network = Network('Network Domain Demonstration')
bus_1 = Bus('BUS-01', Voltage.from_kv(13.8))
bus_2 = Bus('BUS-02', Voltage.from_kv(13.8))
line_1 = DemonstrationBranch('LINE-01')

network.add_bus(bus_1)
network.add_bus(bus_2)
network.add_equipment(line_1)
network.connect(line_1.terminal_from.id, bus_1.id)
network.connect(line_1.terminal_to.id, bus_2.id)

print(f'Network: {network.name}')
print(f'Buses: {len(network.buses)} | Equipment: {len(network.equipment)}')
print(f'Terminals: {len(network.terminals)} | Connections: {len(network.connections)}')

Network: Network Domain Demonstration
Buses: 2 | Equipment: 1
Terminals: 2 | Connections: 2


In [4]:
validator = DefaultNetworkValidationService()
diagnostics = validator.validate_for_calculation(network)
if diagnostics:
    for diagnostic in diagnostics:
        print(diagnostic.severity.value, diagnostic.code, diagnostic.message)
else:
    print('Network structure is ready for the next engineering validation stage.')

Network structure is ready for the next engineering validation stage.


## ND-002 — Electrical Branch Modeling
Este bloque ejecuta modelos de ramas con impedancias serie y admitancias shunt por secuencia. Los parámetros son entradas eléctricas explícitas; todavía no se derivan de geometría de conductores.

In [5]:
from uuid import uuid4
from power_fault_analyzer.domain.entities.terminal import Terminal
from power_fault_analyzer.domain.enums.construction import ConductorMaterial
from power_fault_analyzer.domain.enums.sequence_type import SequenceType
from power_fault_analyzer.domain.value_objects.frequency import Frequency
from power_fault_analyzer.domain.value_objects.sequence_parameters import (
    AdmittancePerLength, ImpedancePerLength,
    SequenceAdmittancePerLengthSet, SequenceImpedancePerLengthSet,
)
from power_fault_analyzer.equipment.transmission import LineParameters, OverheadLine

equipment_id = uuid4()
voltage = Voltage.from_kv(13.8)
terminals = (
    Terminal(equipment_id, 'FROM', voltage),
    Terminal(equipment_id, 'TO', voltage),
)
line_parameters = LineParameters(
    series_impedance_per_km=SequenceImpedancePerLengthSet(
        positive=ImpedancePerLength(0.10, 0.40),
        negative=ImpedancePerLength(0.12, 0.42),
        zero=ImpedancePerLength(0.30, 1.20),
    ),
    shunt_admittance_per_km=SequenceAdmittancePerLengthSet(
        positive=AdmittancePerLength(1e-6, 4e-6),
        negative=AdmittancePerLength(1.2e-6, 4.2e-6),
        zero=AdmittancePerLength(0.5e-6, 2e-6),
    ),
    reference_frequency=Frequency(60.0),
)
overhead_line = OverheadLine(
    id=equipment_id, name='OHL-01', terminals=terminals,
    length_km=10.0, parameters=line_parameters,
    conductor_material=ConductorMaterial.ACSR, bundle_subconductors=2,
)
positive_equivalent = overhead_line.build_sequence_equivalent(SequenceType.POSITIVE)
positive_equivalent


BranchEquivalentModel(series_impedance=Impedance(resistance_ohm=1.0, reactance_ohm=4.0), shunt_admittance_from=Admittance(conductance_siemens=4.9999999999999996e-06, susceptance_siemens=1.9999999999999998e-05), shunt_admittance_to=Admittance(conductance_siemens=4.9999999999999996e-06, susceptance_siemens=1.9999999999999998e-05))

## ND-003 — Transformer Equipment Modeling
Two- and three-winding transformer domain entities. Per-unit leakage data, vector groups, grounding and taps are explicit. ND-005 adds positive/negative sequence network adapters and a two-winding zero-sequence adapter for supported grounding/connection cases. Three-winding zero sequence is deliberately not inferred until an explicit validated external-port model is available.

In [6]:
from power_fault_analyzer.domain.enums.winding_connection import WindingConnection
from power_fault_analyzer.domain.enums.winding_designation import WindingDesignation
from power_fault_analyzer.domain.value_objects.frequency import Frequency
from power_fault_analyzer.domain.value_objects.neutral_grounding import NeutralGrounding
from power_fault_analyzer.domain.value_objects.transformer_impedance import (
    PerUnitImpedance, SequencePerUnitImpedance, TransformerImpedanceBase,
)
from power_fault_analyzer.domain.value_objects.transformer_winding import TransformerWinding
from power_fault_analyzer.domain.value_objects.vector_group import VectorGroup
from power_fault_analyzer.equipment.transformers import TwoWindingTransformer

transformer_id = uuid4()
hv_winding = TransformerWinding(
    WindingDesignation.HIGH_VOLTAGE, 115.0, 40.0, WindingConnection.STAR, NeutralGrounding.solid(), neutral_terminal_available=True
)
lv_winding = TransformerWinding(
    WindingDesignation.LOW_VOLTAGE, 13.8, 40.0, WindingConnection.DELTA, None
)
transformer_terminals = (
    Terminal(transformer_id, 'HV', Voltage.from_kv(115.0)),
    Terminal(transformer_id, 'LV', Voltage.from_kv(13.8)),
)
transformer_base = TransformerImpedanceBase(40.0, 115.0)
transformer_z = SequencePerUnitImpedance(
    transformer_base, PerUnitImpedance(0.01, 0.08),
    PerUnitImpedance(0.012, 0.082), PerUnitImpedance(0.015, 0.09)
)
transformer_2w = TwoWindingTransformer(
    id=transformer_id, name='TR-115-13.8', terminals=transformer_terminals,
    windings=(hv_winding, lv_winding), frequency=Frequency(60.0),
    vector_group=VectorGroup(WindingConnection.STAR, WindingConnection.DELTA, 11),
    sequence_impedances=transformer_z,
)
print(transformer_2w.name, transformer_2w.vector_group_notation)
print('Z1 pu:', transformer_2w.get_sequence_impedance_pu(SequenceType.POSITIVE).value_pu)

TR-115-13.8 YNd11
Z1 pu: (0.01+0.08j)


### Three-winding pairwise impedance conversion
The three pairwise impedances must be on one declared common base. The star conversion is applied separately to each sequence.

In [7]:
from power_fault_analyzer.domain.value_objects.transformer_impedance import (
    PairwiseTransformerImpedances,
)
from power_fault_analyzer.equipment.transformers import ThreeWindingTransformer

three_id = uuid4()
three_windings = (
    TransformerWinding(WindingDesignation.HIGH_VOLTAGE, 220.0, 100.0, WindingConnection.STAR, NeutralGrounding.solid(), neutral_terminal_available=True),
    TransformerWinding(WindingDesignation.MEDIUM_VOLTAGE, 66.0, 60.0, WindingConnection.DELTA, None),
    TransformerWinding(WindingDesignation.LOW_VOLTAGE, 13.8, 40.0, WindingConnection.STAR, NeutralGrounding.ungrounded()),
)
three_terminals = (
    Terminal(three_id, 'HV', Voltage.from_kv(220.0)),
    Terminal(three_id, 'MV', Voltage.from_kv(66.0)),
    Terminal(three_id, 'LV', Voltage.from_kv(13.8)),
)
three_base = TransformerImpedanceBase(100.0, 220.0)
def _sequence_pairwise(r: float, x: float) -> SequencePerUnitImpedance:
    z = PerUnitImpedance(r, x)
    return SequencePerUnitImpedance(three_base, z, z, z)
pairwise = PairwiseTransformerImpedances(
    _sequence_pairwise(0.12, 0.36), _sequence_pairwise(0.16, 0.48),
    _sequence_pairwise(0.10, 0.30),
)
transformer_3w = ThreeWindingTransformer(
    id=three_id, name='TR-220-66-13.8', terminals=three_terminals,
    windings=three_windings, frequency=Frequency(50.0),
    hv_mv_vector_group=VectorGroup(WindingConnection.STAR, WindingConnection.DELTA, 11),
    hv_lv_vector_group=VectorGroup(WindingConnection.STAR, WindingConnection.STAR, 0),
    pairwise_impedances=pairwise,
)
(
    transformer_3w.hv_mv_vector_group_notation,
    transformer_3w.hv_lv_vector_group_notation,
    transformer_3w.build_star_equivalent(SequenceType.POSITIVE),
)

('YNd11',
 'YNy0',
 ThreeWindingStarEquivalent(base=ElectricalImpedanceBase(base_power_mva=100.0, base_voltage_kv=220.0), high_voltage=PerUnitImpedance(resistance_pu=0.09000000000000001, reactance_pu=0.27), medium_voltage=PerUnitImpedance(resistance_pu=0.03, reactance_pu=0.08999999999999997), low_voltage=PerUnitImpedance(resistance_pu=0.07, reactance_pu=0.21000000000000002)))

## ND-004 — Electrical Sources
This section demonstrates Thevenin-type source models and converter current-capability declarations. Source impedances are converted onto an explicit three-phase system base. Missing negative/zero sequence data remain **unspecified**; they are not silently treated as equal to positive sequence or as open circuits.

The induction-generator model in this increment is an initial equivalent only. Its decay time constant is retained as metadata; a time-domain decay solver is not yet implemented. Converter-based generation is represented by sequence current limits, not a fixed Thevenin impedance.

In [8]:
from power_fault_analyzer.domain.enums.converter_control_mode import (
    ConverterControlMode, ConverterCurrentPriority,
)
from power_fault_analyzer.domain.enums.sequence_type import SequenceType
from power_fault_analyzer.domain.enums.source_operating_regime import SourceOperatingRegime
from power_fault_analyzer.domain.enums.source_sequence_status import SourceSequenceStatus
from power_fault_analyzer.domain.value_objects.electrical_impedance_base import ElectricalImpedanceBase
from power_fault_analyzer.domain.value_objects.source_models import (
    ConverterFaultCurrentCapability, ExternalGridShortCircuitData,
    SynchronousMachineSequenceImpedances,
)
from power_fault_analyzer.domain.value_objects.neutral_grounding import NeutralGrounding
from power_fault_analyzer.domain.value_objects.transformer_impedance import PerUnitImpedance
from power_fault_analyzer.domain.value_objects.frequency import Frequency
from power_fault_analyzer.equipment.sources import (
    ConverterBasedGenerator, ExternalGrid, SynchronousGenerator,
)

source_network = Network('Source Model Demonstration')
source_bus = Bus('SOURCE-BUS-13K8', Voltage.from_kv(13.8))
source_network.add_bus(source_bus)

grid_id = uuid4()
grid_terminal = Terminal(grid_id, 'P', Voltage.from_kv(13.8))
grid = ExternalGrid(
    id=grid_id, name='UTILITY-GRID', terminal=grid_terminal,
    frequency=Frequency(60),
    short_circuit_data=ExternalGridShortCircuitData(
        short_circuit_power_mva=500.0, rx_ratio=0.1,
        negative_sequence_impedance_ohm=None,
        zero_sequence_impedance_ohm=None,
    ),
)
source_network.add_equipment(grid)
source_network.connect(grid_terminal.id, source_bus.id)

system_base = ElectricalImpedanceBase(100.0, 13.8)
grid_model = grid.build_source_equivalent(
    system_base, SourceOperatingRegime.SUBTRANSIENT
)
print('External grid Z1 (pu):', grid_model.positive.impedance.value_pu)
print('External grid Z2 status:', grid_model.negative.status.value)
print('External grid Z0 status:', grid_model.zero.status.value)

machine_id = uuid4()
machine_terminal = Terminal(machine_id, 'P', Voltage.from_kv(13.8))
machine_data = SynchronousMachineSequenceImpedances(
    base=ElectricalImpedanceBase(100.0, 13.8),
    positive_subtransient=PerUnitImpedance(0.01, 0.20),
    positive_transient=PerUnitImpedance(0.02, 0.30),
    positive_synchronous=PerUnitImpedance(0.03, 1.50),
    negative=PerUnitImpedance(0.01, 0.18),
    zero=PerUnitImpedance(0.02, 0.10),
)
generator = SynchronousGenerator(
    id=machine_id, name='GENERATOR-01', terminal=machine_terminal,
    frequency=Frequency(60), sequence_impedances=machine_data,
    neutral_grounding=NeutralGrounding.solid(),
)
source_network.add_equipment(generator)
source_network.connect(machine_terminal.id, source_bus.id)
machine_model = generator.build_source_equivalent(
    system_base, SourceOperatingRegime.SUBTRANSIENT
)
print('Generator Z1 subtransient (pu):', machine_model.positive.impedance.value_pu)
print('Generator Z0 status:', machine_model.zero.status.value)

converter_id = uuid4()
converter_terminal = Terminal(converter_id, 'P', Voltage.from_kv(13.8))
converter = ConverterBasedGenerator(
    id=converter_id, name='PV-CONVERTER-01', terminal=converter_terminal,
    frequency=Frequency(60), control_mode=ConverterControlMode.GRID_FOLLOWING,
    current_priority=ConverterCurrentPriority.REACTIVE,
    fault_current_capability=ConverterFaultCurrentCapability(
        base=ElectricalImpedanceBase(10.0, 13.8),
        overall_current_limit_pu=1.2, positive_sequence_limit_pu=1.2,
        negative_sequence_limit_pu=0.2, zero_sequence_limit_pu=0.0,
    ),
)
print('Converter family:', converter.family.value)
print('Converter overall current limit (pu):', converter.fault_current_capability.overall_current_limit_pu)
print('Network equipment:', [item.name for item in source_network.equipment])

External grid Z1 (pu): (0.019900743804199785+0.19900743804199783j)
External grid Z2 status: unspecified
External grid Z0 status: unspecified
Generator Z1 subtransient (pu): (0.01+0.2j)
Generator Z0 status: available
Converter family: converter
Converter overall current limit (pu): 1.2
Network equipment: ['UTILITY-GRID', 'GENERATOR-01']


## ND-005 — Sequence Networks and Ybus

The next cell builds a two-bus 13.8-kV example with an external Thevenin grid and a nominal-π line. It constructs positive, negative and zero sequence networks, then assembles sparse Ybus matrices and Norton injection vectors using SciPy. This is network construction and matrix assembly, not yet a short-circuit solution.

In [9]:
from uuid import uuid4

from power_fault_analyzer.domain.enums.sequence_type import SequenceType
from power_fault_analyzer.domain.value_objects.frequency import Frequency
from power_fault_analyzer.domain.value_objects.voltage import Voltage
from power_fault_analyzer.domain.entities.bus import Bus
from power_fault_analyzer.domain.entities.terminal import Terminal
from power_fault_analyzer.domain.aggregates.network import Network
from power_fault_analyzer.equipment.sources.external_grid import ExternalGrid
from power_fault_analyzer.domain.value_objects.impedance import Impedance
from power_fault_analyzer.domain.value_objects.sequence_parameters import (
    AdmittancePerLength,
    ImpedancePerLength,
    SequenceAdmittancePerLengthSet,
    SequenceImpedancePerLengthSet,
)
from power_fault_analyzer.domain.value_objects.source_models import ExternalGridShortCircuitData
from power_fault_analyzer.equipment.transmission.line import Line
from power_fault_analyzer.equipment.transmission.line_parameters import LineParameters
from power_fault_analyzer.network.sequence_networks import SequenceNetworkBuilder, SystemBaseSet
from power_fault_analyzer.infrastructure.numerical.ybus import SparseYBusAssembler

nd005_network = Network("ND-005 Two-Bus Sequence Study")
nd005_bus_a = Bus("ND005-BUS-A", Voltage.from_kv(13.8))
nd005_bus_b = Bus("ND005-BUS-B", Voltage.from_kv(13.8))
nd005_network.add_bus(nd005_bus_a)
nd005_network.add_bus(nd005_bus_b)

nd005_line_id = uuid4()
nd005_line_terminals = (
    Terminal(nd005_line_id, "FROM", Voltage.from_kv(13.8)),
    Terminal(nd005_line_id, "TO", Voltage.from_kv(13.8)),
)
nd005_line_parameters = LineParameters(
    series_impedance_per_km=SequenceImpedancePerLengthSet(
        positive=ImpedancePerLength(0.10, 0.40),
        negative=ImpedancePerLength(0.12, 0.42),
        zero=ImpedancePerLength(0.30, 1.20),
    ),
    shunt_admittance_per_km=SequenceAdmittancePerLengthSet(
        positive=AdmittancePerLength(1e-6, 4e-6),
        negative=AdmittancePerLength(1.2e-6, 4.2e-6),
        zero=AdmittancePerLength(0.5e-6, 2e-6),
    ),
    reference_frequency=Frequency(60.0),
)
nd005_line = Line(
    id=nd005_line_id,
    name="ND005-LINE-A-B",
    terminals=nd005_line_terminals,
    length_km=8.0,
    parameters=nd005_line_parameters,
)
nd005_network.add_equipment(nd005_line)
nd005_network.connect(nd005_line_terminals[0].id, nd005_bus_a.id)
nd005_network.connect(nd005_line_terminals[1].id, nd005_bus_b.id)

nd005_grid_id = uuid4()
nd005_grid_terminal = Terminal(nd005_grid_id, "P", Voltage.from_kv(13.8))
nd005_grid = ExternalGrid(
    id=nd005_grid_id,
    name="ND005-UTILITY-GRID",
    terminal=nd005_grid_terminal,
    frequency=Frequency(60.0),
    short_circuit_data=ExternalGridShortCircuitData(
        short_circuit_power_mva=500.0,
        rx_ratio=0.10,
        negative_sequence_impedance_ohm=Impedance(0.008, 0.080),
        zero_sequence_impedance_ohm=Impedance(0.020, 0.200),
        prefault_voltage_pu=1.0 + 0.0j,
    ),
)
nd005_network.add_equipment(nd005_grid)
nd005_network.connect(nd005_grid_terminal.id, nd005_bus_a.id)

nd005_system_base = SystemBaseSet.from_network(nd005_network, base_power_mva=100.0)
nd005_sequence_set = SequenceNetworkBuilder(nd005_system_base).build_all(nd005_network)
nd005_assembler = SparseYBusAssembler()
nd005_ybus = {
    sequence: nd005_assembler.assemble(nd005_sequence_set.for_sequence(sequence))
    for sequence in SequenceType
}

nd005_summary = {
    sequence.value: {
        "complete": nd005_sequence_set.for_sequence(sequence).is_complete,
        "nodes": nd005_ybus[sequence].matrix.shape[0],
        "branches": len(nd005_sequence_set.for_sequence(sequence).branches),
        "shunts": len(nd005_sequence_set.for_sequence(sequence).shunts),
        "Ybus_nnz": nd005_ybus[sequence].matrix.nnz,
        "Ybus_pu": nd005_ybus[sequence].to_dense().tolist(),
        "Norton_current_pu": nd005_ybus[sequence].current_injection_pu.tolist(),
        "diagnostics": [item.code for item in nd005_ybus[sequence].diagnostics],
    }
    for sequence in SequenceType
}
nd005_summary


# Positive-sequence matrix factorization is tested independently of fault solving.
nd005_positive_lu = nd005_ybus[SequenceType.POSITIVE].factorize()
print("Positive-sequence sparse LU factorization: OK")
print("No short-circuit boundary-value problem is solved in ND-005.")


Positive-sequence sparse LU factorization: OK
No short-circuit boundary-value problem is solved in ND-005.


## ND-006 — Bus short-circuit fault solver

This section performs bus faults from the ND-005 sparse sequence networks. The solver uses sparse LU reductions to obtain the prefault Thevenin voltage and driving-point impedance at the selected bus. The supported canonical cases are three-phase, A-G, B-C and B-C-G. Fault impedance is per unit on the faulted bus base; reported currents are kA and phase voltages are line-to-neutral kV.

ND-006 is a sequence-network bus-fault solver, not an IEC 60909 implementation. It does not yet solve line-interior faults, load-flow prefault states or nonlinear converter current limits.

In [10]:
from power_fault_analyzer.domain.enums.fault_type import FaultType
from power_fault_analyzer.domain.value_objects.fault_specification import FaultSpecification
from power_fault_analyzer.infrastructure.numerical.short_circuit import ShortCircuitSolver

nd006_solver = ShortCircuitSolver()
nd006_fault_cases = {
    FaultType.THREE_PHASE: 0.0j,
    FaultType.SINGLE_LINE_TO_GROUND: 0.02 + 0.04j,
    FaultType.LINE_TO_LINE: 0.015 + 0.03j,
    FaultType.DOUBLE_LINE_TO_GROUND: 0.01 + 0.025j,
}
nd006_fault_results = {
    fault_type.value: nd006_solver.solve(
        nd005_sequence_set,
        FaultSpecification(
            fault_type=fault_type,
            bus_id=nd005_bus_b.id,
            fault_impedance_pu=fault_impedance,
        ),
        nd005_system_base,
    )
    for fault_type, fault_impedance in nd006_fault_cases.items()
}

nd006_summary = {
    fault_type: {
        "sequence_currents_pu": {
            sequence.value: current
            for sequence, current in result.sequence_currents_pu.items()
        },
        "phase_currents_ka": result.phase_currents_ka,
        "phase_voltages_kv_LN": result.phase_voltages_kv,
        "ground_current_ka": result.ground_current_ka,
        "maximum_phase_current_ka": result.maximum_phase_current_ka,
    }
    for fault_type, result in nd006_fault_results.items()
}
nd006_summary

{'three_phase': {'sequence_currents_pu': {'zero': 0j,
   'positive': (0.1181008864328408-0.5044589548417827j),
   'negative': 0j},
  'phase_currents_ka': ((0.49409839546039164-2.1105037200943513j),
   (-2.074799034113466+0.6273500976093477j),
   (1.580700638653075+1.4831536224850042j)),
  'phase_voltages_kv_LN': ((-1.769125671472879e-15-5.0534107315021207e-17j),
   (8.407990150440627e-16+1.5573748276402262e-15j),
   (9.283266564288165e-16-1.5068407203252057e-15j)),
  'ground_current_ka': 0j,
  'maximum_phase_current_ka': 2.1675698782112263},
 'single_line_to_ground': {'sequence_currents_pu': {'zero': (0.026709940470299665-0.10491061322784949j),
   'positive': (0.026709940470299665-0.10491061322784949j),
   'negative': (0.026709940470299665-0.10491061322784949j)},
  'phase_currents_ka': ((0.33523894175144325-1.3167428432162525j),
   (1.7418140275213439e-16+1.161209351680896e-16j),
   (1.7418140275213439e-16+1.016058182720784e-16j)),
  'phase_voltages_kv_LN': ((0.11307278363827038-0.0246

## ND-007 — Interior line/cable fault location

This study inserts an internal fault node at a normalized position along the existing ND-005 line. The solver preserves the original terminal shunts, splits the series impedance by distance, reuses the ND-006 fault solver, and recovers current and voltage states at both physical line terminals. Terminal currents are reported directed into the line from each bus. This is a lumped sequence-model calculation, not IEC 60909 or a distributed-parameter simulation.

In [11]:
from power_fault_analyzer.domain.value_objects.line_fault_specification import (
    LineFaultSpecification,
)
from power_fault_analyzer.infrastructure.numerical.line_fault import LineFaultSolver

nd007_line_solver = LineFaultSolver()
nd007_fault_location = LineFaultSpecification(
    fault_type=FaultType.THREE_PHASE,
    branch_id=nd005_line_id,
    fraction_from=0.40,
    fault_impedance_pu=0.0j,
)
nd007_result = nd007_line_solver.solve(
    nd005_sequence_set,
    nd007_fault_location,
    nd005_system_base,
)
nd007_summary = {
    "line": nd007_result.line_name,
    "location_fraction_from": nd007_result.fraction_from,
    "distance_from_km": nd007_result.distance_from_km,
    "distance_to_km": nd007_result.distance_to_km,
    "fault_phase_currents_ka": nd007_result.fault_result.phase_currents_ka,
    "from_terminal_currents_ka": nd007_result.from_terminal.phase_currents_into_line_ka,
    "to_terminal_currents_ka": nd007_result.to_terminal.phase_currents_into_line_ka,
    "from_terminal_voltages_kv_LN": nd007_result.from_terminal.phase_voltages_kv,
    "to_terminal_voltages_kv_LN": nd007_result.to_terminal.phase_voltages_kv,
}
nd007_summary

{'line': 'ND005-LINE-A-B',
 'location_fraction_from': 0.4,
 'distance_from_km': 3.2,
 'distance_to_km': 4.8,
 'fault_phase_currents_ka': ((0.9899966032642766-4.589028382435872j),
  (-4.469213459509413+1.4371519831307698j),
  (3.4792168562451375+3.151876399305103j)),
 'from_terminal_currents_ka': ((0.9900245869801303-4.588930135525712j),
  (-4.469142367047298+1.4370786250668652j),
  (3.4791177800671687+3.1518515104588447j)),
 'to_terminal_currents_ka': ((5.848382017906338e-33+2.157980179496553e-34j),
  (-2.7373044433224357e-33-5.172746407517814e-33j),
  (-3.111077574583899e-33+4.9569483895681585e-33j)),
 'from_terminal_voltages_kv_LN': ((6.190755242562484-0.20129343020120483j),
  (-3.2697028454503942-5.260704593570204j),
  (-2.921052397112088+5.461998023771408j)),
 'to_terminal_voltages_kv_LN': ((-4.156726504956859e-17-1.926529278115037e-16j),
  (-1.460586970734275e-16+1.3232477140451922e-16j),
  (1.8762596212299607e-16+6.032815640698438e-17j))}

## ND-008 — IEC 60909-0:2026 network-feeder study

This section uses the existing two-bus 13.8-kV network and runs a maximum three-phase bolted bus fault with one ExternalGrid network feeder. The calculator applies the equivalent voltage factor, supported two-winding transformer corrections, and radial peak/thermal calculations only where the topology supports those methods. The result carries a warning that this is a restricted network-feeder subset, not full IEC 60909 compliance.


In [12]:
from power_fault_analyzer.domain.enums.iec60909 import (
    IEC60909CurrentCase,
    IEC60909LowVoltageTolerance,
)
from power_fault_analyzer.domain.value_objects.iec60909_study import (
    IEC60909StudySpecification,
)
from power_fault_analyzer.infrastructure.numerical.iec60909 import IEC60909Calculator

nd008_study_specification = IEC60909StudySpecification(
    current_case=IEC60909CurrentCase.MAXIMUM,
    fault_duration_s=1.0,
    frequency_hz=60.0,
    low_voltage_tolerance=IEC60909LowVoltageTolerance.PERCENT_10,
)
nd008_result = IEC60909Calculator().calculate(
    nd005_sequence_set,
    FaultSpecification(
        fault_type=FaultType.THREE_PHASE,
        bus_id=nd005_bus_b.id,
        fault_impedance_pu=0.0j,
    ),
    nd005_system_base,
    nd008_study_specification,
)
nd008_summary = {
    "standard_edition": nd008_result.standard_edition,
    "current_case": nd008_result.current_case.value,
    "voltage_factor_c": nd008_result.voltage_factor,
    "initial_symmetrical_current_ka": nd008_result.initial_symmetrical_current_ka,
    "peak_current_factor_kappa": nd008_result.peak_current_factor,
    "peak_current_ka": nd008_result.peak_current_ka,
    "breaking_current_ka": nd008_result.breaking_current_ka,
    "steady_state_current_ka": nd008_result.steady_state_current_ka,
    "thermal_factor_m": nd008_result.thermal_factor_dc,
    "thermal_factor_n": nd008_result.thermal_factor_ac,
    "thermal_equivalent_current_ka": nd008_result.thermal_equivalent_current_ka,
    "joule_integral_ka2s": nd008_result.joule_integral_ka2s,
    "diagnostics": [item.code for item in nd008_result.diagnostics],
}
nd008_summary


{'standard_edition': 'IEC 60909-0:2026',
 'current_case': 'maximum',
 'voltage_factor_c': 1.1,
 'initial_symmetrical_current_ka': 2.3843142857639092,
 'peak_current_factor_kappa': 1.5055121905800581,
 'peak_current_ka': 5.076481118407978,
 'breaking_current_ka': 2.3843142857639092,
 'steady_state_current_ka': 2.3843142857639092,
 'thermal_factor_m': 0.012215683778787213,
 'thermal_factor_n': 1.0,
 'thermal_equivalent_current_ka': 2.3988330957260584,
 'joule_integral_ka2s': 5.754400221150665,
 'diagnostics': ['IEC60909-RESTRICTED-NETWORK-FEEDER-METHOD',
  'IEC60909-POSITIVE-NEGATIVE-SHUNTS-NEGLECTED',
  'IEC60909-LINE-RESISTANCE-REFERENCE-UNVERIFIED',
  'IEC60909-DC-ASYMMETRICAL-BREAKING-NOT-EVALUATED']}

## ND-009 — Protection coordination and short-circuit duty

This section demonstrates phase overcurrent 50/51 relay settings with CT scaling, sampled relay-to-relay grading, and breaker/conductor duty checks using the preceding ND-008 IEC result. Fuse coordination uses an explicitly supplied manufacturer-style time-current point set; the notebook does not invent or claim a specific commercial fuse curve. Coordination margins and all equipment ratings remain study inputs.

In [13]:
from uuid import uuid4

from power_fault_analyzer.domain.enums.protection import (
    CircuitBreakerStandard,
    IECInverseCurveType,
    ProtectionFunction,
)
from power_fault_analyzer.domain.value_objects.protection_devices import (
    CircuitBreaker,
    ConductorThermalLimit,
    CurrentTransformerRatio,
    FuseLink,
    FuseTimeCurrentPoint,
    OvercurrentRelay,
)
from power_fault_analyzer.infrastructure.protection import (
    ProtectionCoordinationAnalyzer,
    ProtectionDutyEvaluator,
)

nd009_duty_evaluator = ProtectionDutyEvaluator()
nd009_downstream_relay = OvercurrentRelay(
    id=uuid4(),
    name="FEEDER-DOWNSTREAM-51",
    current_transformer=CurrentTransformerRatio(1000.0, 5.0),
    function=ProtectionFunction.PHASE,
    pickup_secondary_a=1.0,
    curve_type=IECInverseCurveType.STANDARD_INVERSE,
    time_multiplier=0.10,
)
nd009_upstream_relay = OvercurrentRelay(
    id=uuid4(),
    name="INCOMER-UPSTREAM-51",
    current_transformer=CurrentTransformerRatio(1000.0, 5.0),
    function=ProtectionFunction.PHASE,
    pickup_secondary_a=0.5,
    curve_type=IECInverseCurveType.STANDARD_INVERSE,
    time_multiplier=1.0,
)
nd009_downstream_breaker = CircuitBreaker(
    id=uuid4(),
    name="FEEDER-CB",
    rated_voltage_kv=15.0,
    rated_current_a=630.0,
    rated_short_circuit_breaking_current_ka=50.0,
    standard=CircuitBreakerStandard.IEC_62271_100,
    making_capacity_peak_ka=100.0,
    short_time_withstand_current_ka=30.0,
    short_time_withstand_duration_s=1.0,
    opening_time_s=0.04,
    arcing_time_s=0.02,
)
nd009_coordination = ProtectionCoordinationAnalyzer().coordinate_relay_pair(
    nd009_downstream_relay,
    nd009_upstream_relay,
    minimum_current_a=1500.0,
    maximum_current_a=8000.0,
    minimum_coordination_interval_s=0.30,
    downstream_breaker=nd009_downstream_breaker,
    sample_count=51,
)
nd009_breaker_duty = nd009_duty_evaluator.evaluate_breaker(
    nd009_downstream_breaker,
    nd008_result,
    operating_voltage_kv=13.8,
)
nd009_conductor_duty = nd009_duty_evaluator.evaluate_conductor(
    ConductorThermalLimit(cross_section_mm2=300.0, k_factor_a_sqrt_s_per_mm2=143.0),
    nd008_result,
)
nd009_minimum_fault_sensitivity = nd009_duty_evaluator.evaluate_relay_sensitivity(
    nd009_downstream_relay,
    minimum_fault_current_a=1500.0,
    required_sensitivity_ratio=1.5,
)

nd009_fuse = FuseLink(
    id=uuid4(),
    name="DEMO-FUSE-MANUFACTURER-DATA",
    rated_current_a=200.0,
    rated_voltage_kv=0.4,
    breaking_capacity_ka=50.0,
    characteristic_points=(
        FuseTimeCurrentPoint(200.0, 10.0, 12.0),
        FuseTimeCurrentPoint(2000.0, 1.0, 1.2),
        FuseTimeCurrentPoint(20000.0, 0.1, 0.12),
    ),
)
nd009_fuse_coordination = ProtectionCoordinationAnalyzer().coordinate_fuse_relay(
    nd009_fuse,
    nd009_upstream_relay,
    minimum_current_a=2000.0,
    maximum_current_a=15000.0,
    minimum_coordination_interval_s=0.30,
    sample_count=31,
)

nd009_summary = {
    "relay_pickup_primary_a": nd009_downstream_relay.pickup_primary_a,
    "downstream_relay_time_at_3kA_s": nd009_downstream_relay.operate(3000.0).operating_time_s,
    "relay_coordination_status": nd009_coordination.status.value,
    "relay_coordination_min_margin_s": nd009_coordination.minimum_achieved_margin_s,
    "breaker_duty_status": nd009_breaker_duty.status.value,
    "breaker_checks": {key: value.value for key, value in nd009_breaker_duty.checks.items()},
    "conductor_thermal_status": nd009_conductor_duty.status.value,
    "conductor_thermal_utilization": nd009_conductor_duty.utilization,
    "minimum_fault_sensitivity_status": nd009_minimum_fault_sensitivity.status.value,
    "fuse_relay_coordination_status": nd009_fuse_coordination.status.value,
    "fuse_relay_minimum_margin_s": nd009_fuse_coordination.minimum_achieved_margin_s,
}
nd009_summary

{'relay_pickup_primary_a': 200.0,
 'downstream_relay_time_at_3kA_s': 0.251551745897172,
 'relay_coordination_status': 'pass',
 'relay_coordination_min_margin_s': 1.285611134421114,
 'breaker_duty_status': 'pass',
 'breaker_checks': {'voltage': 'pass',
  'rated_breaking_current': 'pass',
  'icm': 'pass',
  'icw': 'pass'},
 'conductor_thermal_status': 'pass',
 'conductor_thermal_utilization': 0.0031266947153898673,
 'minimum_fault_sensitivity_status': 'pass',
 'fuse_relay_coordination_status': 'pass',
 'fuse_relay_minimum_margin_s': 1.0673563672470647}

## ND-010 — Topology-integrated protection study

ND-010 binds phase/earth overcurrent relays and their circuit breakers to physical terminals of transmission branches and transformers. Protection zones explicitly list protected buses/equipment and primary/backup devices. The topology catalog creates reproducible bus maximum/minimum scenarios and interior-line locations. Bus currents are recovered from the solved sequence-network branch states; interior-line currents are taken from the ND-007 terminal result. The interior-line route is not relabeled as an IEC 60909 case, and remote backup currents outside the faulted line require supplemental measurements until the line solver exposes a complete network-wide branch-current state.

In [14]:
from power_fault_analyzer.domain.enums.fault_type import FaultType
from power_fault_analyzer.domain.enums.protection import (
    CircuitBreakerStandard,
    IECInverseCurveType,
    ProtectionFunction,
)
from power_fault_analyzer.domain.value_objects.protection_devices import (
    CircuitBreaker,
    CurrentTransformerRatio,
    OvercurrentRelay,
)
from power_fault_analyzer.domain.value_objects.protection_network import (
    ProtectionDeviceBinding,
    ProtectionZone,
)
from power_fault_analyzer.infrastructure.protection import (
    ProtectionFaultStudyRunner,
    ProtectionScenarioCatalog,
)
from power_fault_analyzer.network.protection import ProtectionNetworkBuilder

nd010_relay = OvercurrentRelay(
    id=uuid4(),
    name="ND010-LINE-A-PHASE-51",
    current_transformer=CurrentTransformerRatio(600.0, 5.0),
    function=ProtectionFunction.PHASE,
    pickup_secondary_a=1.0,
    curve_type=IECInverseCurveType.STANDARD_INVERSE,
    time_multiplier=0.10,
)
nd010_breaker = CircuitBreaker(
    id=uuid4(),
    name="ND010-LINE-A-CB",
    rated_voltage_kv=15.0,
    rated_current_a=630.0,
    standard=CircuitBreakerStandard.IEC_62271_100,
    rated_short_circuit_breaking_current_ka=25.0,
    opening_time_s=0.05,
    arcing_time_s=0.02,
)
nd010_binding = ProtectionDeviceBinding(
    device=nd010_relay,
    monitored_equipment_id=nd005_line_id,
    monitored_terminal_id=nd005_line_terminals[0].id,
    breaker=nd010_breaker,
)
nd010_zone = ProtectionZone(
    name="ND010-LINE-A-B-PROTECTION-ZONE",
    protected_element_ids=(nd005_line_id,),
    primary_device_ids=(nd010_relay.id,),
)
nd010_protection_model = ProtectionNetworkBuilder().build(
    nd005_network,
    bindings=(nd010_binding,),
    zones=(nd010_zone,),
)
nd010_line_scenarios = ProtectionScenarioCatalog().generate(
    nd005_network,
    fault_types=(FaultType.THREE_PHASE,),
    line_fault_fractions=(0.40,),
    include_bus_faults=False,
    include_line_faults=True,
)
nd010_runner = ProtectionFaultStudyRunner()
nd010_line_measurements = nd010_runner.add_line_fault_measurements(
    nd010_protection_model,
    nd010_line_scenarios[0],
    nd007_result,
)
nd010_assessment = nd010_runner.assess_scenarios(
    nd010_protection_model,
    (nd010_line_measurements,),
)
nd010_summary = {
    "scenario": nd010_assessment.scenarios[0].name,
    "relay_primary_current_a": nd010_assessment.device_operations[0].measured_current_a,
    "relay_picked_up": nd010_assessment.device_operations[0].picked_up,
    "complete_clearing_time_s": nd010_assessment.device_operations[0].clearing_time_s,
    "zone_primary_coverage": nd010_assessment.zone_assessments[0].primary_coverage.value,
    "overall_status": nd010_assessment.status.value,
    "diagnostics": [item.code for item in nd010_assessment.diagnostics],
}
nd010_summary

{'scenario': 'ND005-LINE-A-B:lambda=0.4:three_phase',
 'relay_primary_current_a': 4694.510461332598,
 'relay_picked_up': True,
 'complete_clearing_time_s': 0.2539951223452346,
 'zone_primary_coverage': 'pass',
 'overall_status': 'pass',
 'diagnostics': []}

## ND-011 — Advanced protection functions

This section demonstrates 87L/87T biased differential logic, 67/67N directional supervision, 21 distance loops and a steady-state CT excitation screen. The models use explicit phasor conventions, CT/VT ratios and engineer-entered compensation/settings. They are calculation primitives, not relay type-test certification. See `docs/engineering/advanced_protection_functions.md` for assumptions and exclusions.

In [15]:
from power_fault_analyzer.domain.enums.advanced_protection import (
    DifferentialProtectionKind,
    DirectionalSense,
    DistanceCharacteristic,
)
from power_fault_analyzer.domain.value_objects.advanced_protection import (
    CTExcitationPoint,
    CurrentTransformerPerformanceModel,
    DifferentialMeasurement,
    DifferentialProtectionSettings,
    DifferentialSideSettings,
    DirectionalMeasurement,
    DirectionalOvercurrentSettings,
    DistanceMeasurement,
    DistanceProtectionSettings,
    DistanceZoneSettings,
    VoltageTransformerRatio,
)
from power_fault_analyzer.infrastructure.protection.advanced import (
    CurrentTransformerSaturationEvaluator,
    DifferentialProtectionEvaluator,
    DirectionalOvercurrentEvaluator,
    DistanceProtectionEvaluator,
)

nd011_ct_ratio = CurrentTransformerRatio(600.0, 5.0)
nd011_diff = DifferentialProtectionSettings(
    name="ND011-87L-LINE-AB",
    kind=DifferentialProtectionKind.LINE,
    sides=(
        DifferentialSideSettings("FROM", 1000.0, nd011_ct_ratio),
        DifferentialSideSettings("TO", 1000.0, nd011_ct_ratio),
    ),
    minimum_pickup_pu=0.20,
    low_slope=0.30,
    breakpoint_restraint_pu=2.0,
    high_slope=0.50,
    unrestrained_pickup_pu=8.0,
    second_harmonic_block_ratio=0.15,
)
nd011_diff_evaluator = DifferentialProtectionEvaluator()
nd011_through_result = nd011_diff_evaluator.evaluate(
    nd011_diff,
    DifferentialMeasurement(
        {
            "FROM": (1000 + 0j, -500 - 500 * 3**0.5 * 1j, -500 + 500 * 3**0.5 * 1j),
            "TO": (-1000 + 0j, 500 + 500 * 3**0.5 * 1j, 500 - 500 * 3**0.5 * 1j),
        }
    ),
)
nd011_internal_result = nd011_diff_evaluator.evaluate(
    nd011_diff,
    DifferentialMeasurement({"FROM": (1500 + 0j, 0j, 0j), "TO": (0j, 0j, 0j)}),
)
{
    "87L_through_fault_operated": nd011_through_result.operated,
    "87L_internal_fault_operated": nd011_internal_result.operated,
    "87L_internal_A_differential_pu": nd011_internal_result.phases[0].differential_current_pu,
}

{'87L_through_fault_operated': False,
 '87L_internal_fault_operated': True,
 '87L_internal_A_differential_pu': 1.5}

### 67 / 67N — Directional overcurrent

The existing 50/51 relay characteristic supplies the current pickup/time. The directional element supervises it with a measured operating-current phasor and polarizing-voltage phasor. The selected MTA and forward/reverse convention are explicit.

In [16]:
nd011_67_relay = OvercurrentRelay(
    id=uuid4(),
    name="ND011-67-PHASE",
    current_transformer=nd011_ct_ratio,
    function=ProtectionFunction.PHASE,
    pickup_secondary_a=1.0,
    curve_type=IECInverseCurveType.STANDARD_INVERSE,
    time_multiplier=0.10,
)
nd011_67 = DirectionalOvercurrentSettings(
    relay=nd011_67_relay,
    maximum_torque_angle_deg=0.0,
    minimum_polarizing_voltage_v=10.0,
    sense=DirectionalSense.FORWARD,
)
nd011_67_evaluator = DirectionalOvercurrentEvaluator()
nd011_67_forward = nd011_67_evaluator.evaluate(
    nd011_67, DirectionalMeasurement(1500 + 0j, 100 + 0j)
)
nd011_67_reverse = nd011_67_evaluator.evaluate(
    nd011_67, DirectionalMeasurement(-1500 + 0j, 100 + 0j)
)
{
    "forward_fault_operated": nd011_67_forward.operated,
    "forward_torque": nd011_67_forward.directional_torque,
    "reverse_fault_operated": nd011_67_reverse.operated,
    "reverse_decision": nd011_67_reverse.directional_decision.value,
}

{'forward_fault_operated': True,
 'forward_torque': 1.0,
 'reverse_fault_operated': False,
 'reverse_decision': 'reverse'}

### 21 — Distance impedance zones

The measurement below is supplied in primary phasors. The evaluator converts to relay-secondary volts/amperes using the entered VT/CT ratios and evaluates phase-ground and phase-phase loops against the configured impedance zones.

In [17]:
nd011_vt_ratio = VoltageTransformerRatio(
    13_800.0 / 3**0.5,
    110.0 / 3**0.5,
)
nd011_21 = DistanceProtectionSettings(
    name="ND011-21-LINE-AB",
    current_transformer=nd011_ct_ratio,
    voltage_transformer=nd011_vt_ratio,
    ground_compensation_factor=0.0j,
    zones=(
        DistanceZoneSettings(
            1,
            DistanceCharacteristic.MHO,
            0.10,
            reach_impedance_secondary_ohm=12.0 + 24.0j,
        ),
        DistanceZoneSettings(
            2,
            DistanceCharacteristic.MHO,
            0.40,
            reach_impedance_secondary_ohm=24.0 + 48.0j,
        ),
    ),
)
nd011_i_primary = 1200.0 + 0j
nd011_v_secondary = (5.0 + 10.0j) * (nd011_i_primary / nd011_ct_ratio.ratio)
nd011_measurement_21 = DistanceMeasurement(
    phase_voltages_primary_v=(nd011_v_secondary * nd011_vt_ratio.ratio, 0j, 0j),
    phase_currents_primary_a=(nd011_i_primary, 0j, 0j),
)
nd011_21_result = DistanceProtectionEvaluator().evaluate(nd011_21, nd011_measurement_21)
nd011_ag_loop = next(item for item in nd011_21_result.loops if item.loop.value == "a_ground")
{
    "distance_operated": nd011_21_result.operated,
    "fastest_zone": nd011_21_result.fastest_zone_number,
    "AG_apparent_impedance_secondary_ohm": nd011_ag_loop.apparent_impedance_secondary_ohm,
}

{'distance_operated': True,
 'fastest_zone': 1,
 'AG_apparent_impedance_secondary_ohm': (5+10j)}

### CT — Steady-state excitation screen

The calculation compares the symmetrical steady-state secondary voltage demand with the entered knee-point voltage and interpolates the manufacturer excitation curve only inside its supplied domain. Transient DC-offset and remanence saturation remain `NOT_EVALUATED`.

In [18]:
nd011_ct_model = CurrentTransformerPerformanceModel(
    name="ND011-CT-PX-600-5",
    current_transformer=nd011_ct_ratio,
    secondary_winding_resistance_ohm=0.20,
    burden_impedance_ohm=0.50 + 0.10j,
    knee_point_voltage_v=100.0,
    excitation_curve=(
        CTExcitationPoint(1.0, 0.01),
        CTExcitationPoint(10.0, 0.05),
        CTExcitationPoint(100.0, 1.0),
        CTExcitationPoint(110.0, 2.0),
    ),
    rated_frequency_hz=60.0,
)
nd011_ct_result = CurrentTransformerSaturationEvaluator().evaluate(
    nd011_ct_model,
    1000.0,
    system_frequency_hz=60.0,
)
{
    "steady_state_screen": nd011_ct_result.status.value,
    "required_secondary_voltage_V": nd011_ct_result.required_secondary_voltage_v,
    "knee_point_voltage_V": nd011_ct_result.knee_point_voltage_v,
    "transient_status": nd011_ct_result.transient_status.value,
    "diagnostics": list(nd011_ct_result.diagnostics),
}

{'steady_state_screen': 'pass',
 'required_secondary_voltage_V': 5.892556509887895,
 'knee_point_voltage_V': 100.0,
 'transient_status': 'not_evaluated',
 'diagnostics': ['Transient DC-offset, remanence and flux integration are not evaluated by this AC screen.']}

## ND-012 — Transient CT, relay and breaker dynamics

This section integrates the CT flux-linkage ODE, remanent-flux initial condition, exponentially decaying DC offset, sliding-RMS 50/51 timer and ideal per-pole current-zero interruption estimate. The CT excitation curve is synthetic study data for demonstration, not a manufacturer certificate.

In [19]:
import math

from power_fault_analyzer.domain.value_objects.transient_protection import (
    CTTransientSimulationSettings,
    RelayTransientSettings,
    ThreePhaseCurrentSample,
    BreakerTransientSettings,
)
from power_fault_analyzer.infrastructure.protection.transient import (
    CurrentTransformerTransientSimulator,
    OvercurrentRelayTransientEvaluator,
    CircuitBreakerTransientEvaluator,
)

nd012_ct_transient_settings = CTTransientSimulationSettings(
    primary_current_rms_a=1200.0,
    system_frequency_hz=60.0,
    duration_s=0.12,
    time_step_s=0.0001,
    dc_offset_initial_pu=0.5,
    dc_time_constant_s=0.04,
    inception_angle_rad=0.4,
    remanent_flux_pu=0.25,
)
nd012_ct_transient = CurrentTransformerTransientSimulator().simulate(
    nd011_ct_model, nd012_ct_transient_settings
)
nd012_dynamic_relay = OvercurrentRelay(
    id=uuid4(),
    name="ND012-DYNAMIC-51",
    current_transformer=nd011_ct_ratio,
    function=ProtectionFunction.PHASE,
    pickup_secondary_a=5.0,
    curve_type=IECInverseCurveType.STANDARD_INVERSE,
    time_multiplier=0.001,
)
nd012_relay_transient = OvercurrentRelayTransientEvaluator().evaluate_ct_output(
    nd012_dynamic_relay,
    nd012_ct_transient,
    RelayTransientSettings(
        system_frequency_hz=60.0,
        rms_window_cycles=1.0,
        output_delay_s=0.002,
    ),
)
{
    "CT_status": nd012_ct_transient.status.value,
    "CT_peak_flux_pu": nd012_ct_transient.peak_flux_pu,
    "CT_current_transfer_error": nd012_ct_transient.current_transfer_error_ratio,
    "relay_operated": nd012_relay_transient.operated,
    "relay_trip_stage": (None if nd012_relay_transient.trip_stage is None else nd012_relay_transient.trip_stage.value),
    "relay_trip_command_s": nd012_relay_transient.trip_command_time_s,
    "relay_output_time_s": nd012_relay_transient.trip_output_time_s,
}


{'CT_status': 'pass',
 'CT_peak_flux_pu': 0.7321012457456338,
 'CT_current_transfer_error': 0.06923333921935798,
 'relay_operated': True,
 'relay_trip_stage': '51_time_inverse',
 'relay_trip_command_s': 0.0254,
 'relay_output_time_s': 0.0274}

In [20]:
nd012_breaker = CircuitBreaker(
    id=uuid4(),
    name="ND012-MV-BREAKER",
    rated_voltage_kv=13.8,
    rated_current_a=1250.0,
    standard=CircuitBreakerStandard.IEC_62271_100,
    rated_short_circuit_breaking_current_ka=25.0,
    opening_time_s=0.05,
    arcing_time_s=0.02,
)
nd012_peak_primary = math.sqrt(2.0) * nd012_ct_transient_settings.primary_current_rms_a
nd012_omega = 2.0 * math.pi * nd012_ct_transient_settings.system_frequency_hz
nd012_three_phase_waveform = tuple(
    ThreePhaseCurrentSample(
        sample.time_s,
        (
            nd012_peak_primary * math.sin(nd012_omega * sample.time_s + 0.4),
            nd012_peak_primary * math.sin(nd012_omega * sample.time_s + 0.4 - 2.0 * math.pi / 3.0),
            nd012_peak_primary * math.sin(nd012_omega * sample.time_s + 0.4 + 2.0 * math.pi / 3.0),
        ),
    )
    for sample in nd012_ct_transient.samples
)
nd012_breaker_transient = CircuitBreakerTransientEvaluator().evaluate(
    nd012_breaker,
    nd012_relay_transient,
    nd012_three_phase_waveform,
    BreakerTransientSettings(
        pole_scatter_s_by_phase=(0.0, 0.001, 0.002),
        maximum_arc_duration_s=0.025,
    ),
)
{
    "breaker_status": nd012_breaker_transient.status.value,
    "total_clearing_time_s": nd012_breaker_transient.total_clearing_time_s,
    "pole_current_zero_times_s": {pole.phase: pole.current_zero_time_s for pole in nd012_breaker_transient.poles},
    "diagnostics": list(nd012_breaker_transient.diagnostics),
}


{'breaker_status': 'pass',
 'total_clearing_time_s': 0.059650078147908396,
 'pole_current_zero_times_s': {'A': 0.08227229826351146,
  'B': 0.07949452150943834,
  'C': 0.0850500781479084},
 'diagnostics': ['Ideal interruption at first natural current zero; TRV and restrike are not modeled.']}

## ND-013 — Advanced sequence-network models

This increment adds explicit three-winding zero-sequence port equivalents, mutually coupled line branches, source-superposition reporting, and explicit sequence-current phasors for converter sources. The same fixed converter phasors are accepted by the bus-fault solver after capability and connected-bus validation. These are explicit engineering inputs/models; the software does not infer transformer core paths, geometric line coupling, or converter control laws from incomplete nameplate data.


In [21]:
from uuid import uuid4 as _nd013_uuid4

import numpy as _nd013_np

from power_fault_analyzer.domain.enums.fault_type import FaultType as _ND013FaultType
from power_fault_analyzer.domain.enums.sequence_equipment_kind import SequenceEquipmentKind as _ND013BranchKind
from power_fault_analyzer.domain.enums.sequence_source_kind import SequenceSourceKind as _ND013SourceKind
from power_fault_analyzer.domain.enums.sequence_type import SequenceType as _ND013Sequence
from power_fault_analyzer.domain.value_objects.converter_current_command import (
    ConverterSequenceCurrentCommand as _ND013ConverterCommand,
)
from power_fault_analyzer.domain.value_objects.electrical_impedance_base import (
    ElectricalImpedanceBase as _ND013Base,
)
from power_fault_analyzer.domain.value_objects.fault_specification import (
    FaultSpecification as _ND013Fault,
)
from power_fault_analyzer.domain.value_objects.source_models import (
    ConverterFaultCurrentCapability as _ND013Capability,
)
from power_fault_analyzer.infrastructure.numerical.short_circuit import (
    ShortCircuitSolver as _ND013FaultSolver,
)
from power_fault_analyzer.infrastructure.numerical.source_contributions import (
    ConverterCurrentInjectionSolver as _ND013ConverterSolver,
    SourceVoltageContributionCalculator as _ND013SourceContributionCalculator,
)
from power_fault_analyzer.infrastructure.numerical.ybus import (
    SparseYBusAssembler as _ND013Assembler,
)
from power_fault_analyzer.network.sequence_networks.models import (
    SequenceBranch as _ND013Branch,
    SequenceMultiportElement as _ND013Multiport,
    SequenceMutualCoupling as _ND013Mutual,
    SequenceNetwork as _ND013Network,
    SequenceNetworkSet as _ND013NetworkSet,
    SequenceNode as _ND013Node,
    SequenceShunt as _ND013Shunt,
    SystemBaseSet as _ND013SystemBase,
)

# 1) Reciprocal 3-port zero-sequence transformer impedance matrix.
_nd013_ports = tuple(_nd013_uuid4() for _ in range(3))
_nd013_z0 = (
    (0.12 + 0.35j, 0.015 + 0.04j, 0.01 + 0.025j),
    (0.015 + 0.04j, 0.16 + 0.42j, 0.012 + 0.03j),
    (0.01 + 0.025j, 0.012 + 0.03j, 0.20 + 0.50j),
)
_nd013_multiport = _ND013Multiport(
    id=_nd013_uuid4(),
    name="TR3:ZERO-PORTS",
    node_ids=_nd013_ports,
    impedance_matrix_pu=_nd013_z0,
)
_nd013_multiport_network = _ND013Network(
    network_id=_nd013_uuid4(),
    sequence=_ND013Sequence.ZERO,
    nodes=tuple(
        _ND013Node(node, f"PORT-{index}")
        for index, node in enumerate(_nd013_ports)
    ),
    branches=(),
    shunts=(),
    multiport_elements=(_nd013_multiport,),
)
_nd013_multiport_ybus = _ND013Assembler().assemble(_nd013_multiport_network)

# 2) Explicit zero-sequence mutual coupling between two parallel line circuits.
_nd013_a, _nd013_b = _nd013_uuid4(), _nd013_uuid4()
_nd013_line_1, _nd013_line_2 = _nd013_uuid4(), _nd013_uuid4()
_nd013_coupled_network = _ND013Network(
    network_id=_nd013_uuid4(),
    sequence=_ND013Sequence.ZERO,
    nodes=(_ND013Node(_nd013_a, "BUS-A"), _ND013Node(_nd013_b, "BUS-B")),
    branches=(
        _ND013Branch(
            _nd013_line_1,
            "LINE-1",
            _nd013_a,
            _nd013_b,
            0.10 + 0.40j,
            equipment_kind=_ND013BranchKind.LINE,
        ),
        _ND013Branch(
            _nd013_line_2,
            "LINE-2",
            _nd013_a,
            _nd013_b,
            0.12 + 0.42j,
            equipment_kind=_ND013BranchKind.LINE,
        ),
    ),
    shunts=(),
    mutual_couplings=(
        _ND013Mutual(
            _ND013Sequence.ZERO,
            _nd013_line_1,
            _nd013_line_2,
            0.03 + 0.10j,
        ),
    ),
)
_nd013_coupled_ybus = _ND013Assembler().assemble(_nd013_coupled_network)

# 3) Superposition of two independent Thevenin-source contributions at one bus.
_nd013_source_bus = _nd013_uuid4()
_nd013_source_1, _nd013_source_2 = _nd013_uuid4(), _nd013_uuid4()
_nd013_source_network = _ND013Network(
    network_id=_nd013_uuid4(),
    sequence=_ND013Sequence.POSITIVE,
    nodes=(_ND013Node(_nd013_source_bus, "MULTI-SOURCE-BUS"),),
    branches=(),
    shunts=(
        _ND013Shunt(
            _nd013_source_1,
            "GRID-A",
            _nd013_source_bus,
            2.0 + 0j,
            1.0 + 0j,
            _ND013SourceKind.NETWORK_FEEDER,
        ),
        _ND013Shunt(
            _nd013_source_2,
            "GRID-B",
            _nd013_source_bus,
            3.0 + 0j,
            0.9 + 0j,
            _ND013SourceKind.NETWORK_FEEDER,
        ),
    ),
)
_nd013_source_ybus = _ND013Assembler().assemble(_nd013_source_network)
_nd013_contribution = _ND013SourceContributionCalculator().calculate(
    _nd013_source_ybus,
    _nd013_source_bus,
)

# 4) Explicit current-command converter injection, checked against sequence and phase limits.
_nd013_converter_id = _nd013_uuid4()
_nd013_converter_bus = _nd013_uuid4()
_nd013_network_id = _nd013_uuid4()
_nd013_base = _ND013Base(100.0, 13.8)
_nd013_capability = _ND013Capability(_nd013_base, 1.2, 1.0, 0.2, 0.1)
_nd013_system_base = _ND013SystemBase(100.0, {_nd013_converter_bus: 13.8})
_nd013_converter_command = _ND013ConverterCommand(
    _nd013_converter_id,
    _nd013_converter_bus,
    _nd013_capability,
    {
        _ND013Sequence.POSITIVE: 0.5 + 0j,
        _ND013Sequence.NEGATIVE: 0.0j,
        _ND013Sequence.ZERO: 0.0j,
    },
)


def _nd013_converter_sequence(sequence):
    return _ND013Network(
        _nd013_network_id,
        sequence,
        (_ND013Node(_nd013_converter_bus, "CONVERTER-BUS"),),
        (),
        (
            _ND013Shunt(
                _nd013_uuid4(),
                "GRID-THEVENIN",
                _nd013_converter_bus,
                10.0 + 0j,
                1.0 + 0j if sequence is _ND013Sequence.POSITIVE else 0.0j,
            ),
        ),
        nonlinear_source_ids=(_nd013_converter_id,),
        converter_capabilities={_nd013_converter_id: _nd013_capability},
        converter_bus_ids={_nd013_converter_id: _nd013_converter_bus},
    )


_nd013_converter_networks = _ND013NetworkSet(
    positive=_nd013_converter_sequence(_ND013Sequence.POSITIVE),
    negative=_nd013_converter_sequence(_ND013Sequence.NEGATIVE),
    zero=_nd013_converter_sequence(_ND013Sequence.ZERO),
)
_nd013_converter_operating_points = _ND013ConverterSolver().solve_all(
    _nd013_converter_networks,
    _nd013_system_base,
    (_nd013_converter_command,),
)

# 5) The same fixed converter phasor contributes to the three-phase bus-fault result.
_nd013_fault_result = _ND013FaultSolver().solve(
    _nd013_converter_networks,
    _ND013Fault(_ND013FaultType.THREE_PHASE, _nd013_converter_bus),
    _nd013_system_base,
    converter_current_commands=(_nd013_converter_command,),
)
_nd013_np.testing.assert_allclose(
    _nd013_fault_result.sequence_currents_pu[_ND013Sequence.POSITIVE],
    10.5 + 0j,
)

print(
    "3-port zero-sequence Ybus determinant:",
    abs(_nd013_np.linalg.det(_nd013_multiport_ybus.to_dense())),
)
print("Coupled zero-sequence Ybus:", _nd013_coupled_ybus.to_dense())
print("Source voltage contributions:", dict(_nd013_contribution.contributions_pu))
print("Reconstructed prefault voltage:", _nd013_contribution.reconstructed_voltage_pu)
print(
    "Converter sequence bus voltages:",
    {
        key.value: value.node_voltages_pu[_nd013_converter_bus]
        for key, value in _nd013_converter_operating_points.items()
    },
)
print(
    "3-phase bus fault current incl. fixed converter command (pu):",
    _nd013_fault_result.sequence_currents_pu[_ND013Sequence.POSITIVE],
)


3-port zero-sequence Ybus determinant: 11.372384298078275
Coupled zero-sequence Ybus: [[ 0.99791833-3.64993098j -0.99791833+3.64993098j]
 [-0.99791833+3.64993098j  0.99791833-3.64993098j]]
Source voltage contributions: {UUID('a782b754-58ac-4d3d-9df7-57630c130ce3'): (0.4+0j), UUID('7aa5bcc4-3b0d-438d-86be-7fe4da60b0c9'): (0.54+0j)}
Reconstructed prefault voltage: (0.9400000000000001+0j)
Converter sequence bus voltages: {'zero': 0j, 'positive': (1.05+0j), 'negative': 0j}
3-phase bus fault current incl. fixed converter command (pu): (10.5+0j)


## ND-016 — Engineering Validation & Benchmarking

This section runs four deterministic closed-form symmetrical-component reference cases, quantity-level comparisons, unit/base and fault-boundary audits, and a finite-difference sensitivity refinement study. These academic references are **not IEC 60909 normative benchmarks**. No external ETAP/DIgSILENT/PSS/E result is claimed unless a provenance-labelled export is supplied to the CLI.

In [22]:
from power_fault_analyzer.validation.cli import run_validation

nd016_validation_report = run_validation()
nd016_benchmark = nd016_validation_report["benchmark"]
nd016_audits = nd016_validation_report["electrical_audits"]
nd016_sensitivity = nd016_validation_report["sensitivity"]

print(
    f"Analytical cases: {nd016_benchmark['passed_cases']}/{nd016_benchmark['total_cases']}"
)
print(
    f"Quantity comparisons: {nd016_benchmark['passed_comparisons']}"
    f"/{nd016_benchmark['total_comparisons']}"
)
print(f"Electrical audits passed: {sum(item['passed'] for item in nd016_audits)}/{len(nd016_audits)}")
print(f"Sensitivity refinement: {'PASS' if nd016_sensitivity['passed'] else 'FAIL'}")
print(
    "External tool comparison:",
    nd016_validation_report["external_tool_comparison"]["status"],
)


Analytical cases: 4/4
Quantity comparisons: 44/44
Electrical audits passed: 4/4
Sensitivity refinement: PASS
External tool comparison: not_performed


### Interpretation and release gates

- A direct sparse linear solution has no iterative nonlinear load-flow convergence loop; iterative convergence is therefore not applicable to this solver path.
- The implemented sensitivity sweep is restricted to the positive-sequence Thevenin impedance of the declared single-source case.
- Third-party tool comparison remains pending until actual tool output, version and case setup are supplied.
- Passing these checks is evidence for the covered cases only; it is not a blanket IEC 60909 conformity claim or an executable-release certificate.